# Seaborn Tutorial

A hands-on, runnable reference for Seaborn — the statistical visualization library built on top of Matplotlib and designed to work directly with pandas DataFrames. Run each code cell as you go; every plot renders inline.

All examples here build their own small, synthetic datasets with NumPy/pandas (fixed random seeds), so the notebook runs the same way with or without an internet connection — except the one cell in the "Built-in Datasets" section that deliberately demonstrates `sns.load_dataset()`.

In [ ]:
# pip install seaborn   (uncomment to install)

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print(sns.__version__)
sns.set_theme()   # applies seaborn's nicer default look to every plot below

## 1. Seaborn Introduction

- Seaborn is built **on top of Matplotlib** — every seaborn plot is a Matplotlib figure underneath, so `plt.title()`, `plt.show()`, `plt.savefig()`, etc. all still work.
- It's designed around **pandas DataFrames**: instead of passing raw arrays, you pass a DataFrame plus column *names*, and seaborn handles grouping, coloring, and statistics for you.
- Two families of plotting functions:
  - **Axes-level** functions (`scatterplot`, `histplot`, `boxplot`, ...) draw onto a single Matplotlib Axes — combine with `plt.subplots()` for custom layouts.
  - **Figure-level** functions (`relplot`, `displot`, `catplot`, ...) manage their own Figure and can automatically create a grid of subplots (facets) via `col=`/`row=`.

In [ ]:
# The conventional import & setup
import seaborn as sns
sns.set_theme(style="whitegrid")

# A tiny synthetic DataFrame used throughout this notebook
rng = np.random.default_rng(42)
df = pd.DataFrame({
    "day": rng.choice(["Thu", "Fri", "Sat", "Sun"], size=200),
    "total_bill": rng.normal(20, 8, 200).round(2),
    "tip": rng.normal(3, 1.2, 200).round(2),
    "size": rng.integers(1, 6, 200),
    "smoker": rng.choice(["Yes", "No"], size=200, p=[0.3, 0.7]),
    "time": rng.choice(["Lunch", "Dinner"], size=200),
})
df["total_bill"] = df["total_bill"].clip(lower=5)
df["tip"] = df["tip"].clip(lower=0.5)
print(df.head())
print(df.shape)

## 2. Seaborn Getting Started

The most common first plot: `sns.scatterplot(data=df, x=..., y=...)`. Notice you pass **column names**, not raw arrays — seaborn reads them from the DataFrame.

In [ ]:
sns.scatterplot(data=df, x="total_bill", y="tip")
plt.title("Tip vs Total Bill")
plt.show()
plt.close()

## 3. Seaborn Distribution Plots

- `sns.histplot()` — a histogram, optionally with a smoothed density curve (`kde=True`).
- `sns.kdeplot()` — just the smoothed density curve (Kernel Density Estimate).
- `sns.displot()` — the figure-level equivalent, useful for faceting a distribution across categories with `col=`.
- `sns.rugplot()` — draws a small tick for every individual data point along an axis.

In [ ]:
sns.histplot(data=df, x="total_bill", bins=20)
plt.title("Histogram of Total Bill")
plt.show()
plt.close()

sns.histplot(data=df, x="total_bill", kde=True, color="teal")
plt.title("Histogram + KDE")
plt.show()
plt.close()

sns.kdeplot(data=df, x="total_bill", fill=True)
plt.title("KDE Only")
plt.show()
plt.close()

# Figure-level: facet the distribution by smoker status
g = sns.displot(data=df, x="total_bill", col="smoker", kde=True, height=4)
g.figure.suptitle("Total Bill Distribution by Smoker Status", y=1.05)
plt.show()
plt.close("all")

## 4. Seaborn Scatter Plot

`sns.scatterplot()` can map extra DataFrame columns to visual channels: `hue` (color), `size` (point size), and `style` (marker shape) — encoding up to 3 extra dimensions on a single 2D plot.

In [ ]:
sns.scatterplot(data=df, x="total_bill", y="tip", hue="time")
plt.title("Colored by Time of Day")
plt.show()
plt.close()

sns.scatterplot(data=df, x="total_bill", y="tip", hue="time", size="size", style="smoker")
plt.title("Hue + Size + Style Combined")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()
plt.close()

## 5. Seaborn Line Plot

`sns.lineplot()` connects points in order along `x`, and — when multiple `y` values share the same `x` — automatically aggregates them (mean, by default) and draws a **confidence interval band** around the line.

In [ ]:
rng = np.random.default_rng(0)
time_df = pd.DataFrame({
    "hour": np.tile(np.arange(24), 10),
    "visitors": np.concatenate([
        50 + 30 * np.sin(np.linspace(0, 2 * np.pi, 24)) + rng.normal(0, 8, 24)
        for _ in range(10)
    ]),
})

sns.lineplot(data=time_df, x="hour", y="visitors")
plt.title("Visitors by Hour (line + 95% confidence band)")
plt.show()
plt.close()

# Multiple lines via hue
time_df["day_type"] = np.tile(np.repeat(["Weekday", "Weekend"], 12), 10)[: len(time_df)]
sns.lineplot(data=time_df, x="hour", y="visitors", hue="day_type", marker="o")
plt.show()
plt.close()

## 6. Seaborn Bar Plot

`sns.barplot()` shows a **point estimate** (mean, by default) for a numeric column grouped by a categorical column, with error bars representing uncertainty — unlike `plt.bar()`, you don't pre-aggregate the data yourself.

In [ ]:
sns.barplot(data=df, x="day", y="total_bill")
plt.title("Average Total Bill by Day")
plt.show()
plt.close()

# Split further by a second category with hue
sns.barplot(data=df, x="day", y="total_bill", hue="time")
plt.title("Average Total Bill by Day and Time")
plt.show()
plt.close()

## 7. Seaborn Count Plot

`sns.countplot()` counts the number of rows in each category — it's the categorical equivalent of a histogram, and you don't pass a `y` at all (seaborn computes the counts).

In [ ]:
sns.countplot(data=df, x="day")
plt.title("Number of Visits per Day")
plt.show()
plt.close()

sns.countplot(data=df, x="day", hue="smoker")
plt.title("Visits per Day, Split by Smoker Status")
plt.show()
plt.close()

## 8. Seaborn Box Plot

A box plot summarizes a distribution's median, quartiles (the box), and outliers (individual points beyond the "whiskers") — great for comparing distributions across categories at a glance.

In [ ]:
sns.boxplot(data=df, x="day", y="total_bill")
plt.title("Total Bill Distribution by Day")
plt.show()
plt.close()

sns.boxplot(data=df, x="day", y="total_bill", hue="time")
plt.show()
plt.close()

## 9. Seaborn Violin Plot

A violin plot combines a box plot with a KDE, mirrored on both sides — it shows the full shape of the distribution (e.g. whether it's bimodal), not just its quartiles.

In [ ]:
sns.violinplot(data=df, x="day", y="total_bill")
plt.title("Total Bill Distribution Shape by Day")
plt.show()
plt.close()

# split=True mirrors two hue levels onto opposite halves of each violin
sns.violinplot(data=df, x="day", y="total_bill", hue="smoker", split=True)
plt.show()
plt.close()

## 10. Seaborn Strip & Swarm Plot

- `sns.stripplot()` plots every individual observation as a point along a categorical axis (with a little random jitter so points don't overlap).
- `sns.swarmplot()` does the same but arranges points so they never overlap at all — clearer for smaller datasets, slower for large ones.

In [ ]:
sns.stripplot(data=df, x="day", y="total_bill", jitter=True)
plt.title("Every Observation (jittered)")
plt.show()
plt.close()

sns.swarmplot(data=df.sample(60, random_state=1), x="day", y="total_bill")
plt.title("Every Observation (non-overlapping)")
plt.show()
plt.close()

# Combining a swarm on top of a box plot is a common, very informative pattern
sns.boxplot(data=df, x="day", y="total_bill", color="lightgray")
sns.stripplot(data=df, x="day", y="total_bill", color="black", alpha=0.4, jitter=True)
plt.title("Box Plot + Individual Points")
plt.show()
plt.close()

## 11. Seaborn Pair Plot

`sns.pairplot()` plots every numeric column against every other numeric column in a grid — histograms/KDEs on the diagonal, scatter plots off the diagonal. It's the fastest way to eyeball relationships across an entire dataset.

In [ ]:
g = sns.pairplot(df[["total_bill", "tip", "size", "time"]], hue="time")
g.figure.suptitle("Pairwise Relationships", y=1.02)
plt.show()
plt.close("all")

## 12. Seaborn Heatmap

`sns.heatmap()` colors a 2D matrix by value — the most common use is visualizing a **correlation matrix** (`df.corr()`), with `annot=True` printing the actual numbers on top of each cell.

In [ ]:
numeric_df = df[["total_bill", "tip", "size"]]
corr = numeric_df.corr()
print(corr)

sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()
plt.close()

# A heatmap of raw (non-correlation) data too, e.g. a pivot table
pivot = df.pivot_table(values="tip", index="day", columns="time", aggfunc="mean")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Average Tip by Day and Time")
plt.show()
plt.close()

## 13. Seaborn Regression Plot

- `sns.regplot()` (axes-level) draws a scatter plot **plus** a fitted linear regression line with a confidence band.
- `sns.lmplot()` (figure-level) does the same but can facet across `col=`/`row=`/`hue=` to fit a separate regression line per group.

In [ ]:
sns.regplot(data=df, x="total_bill", y="tip")
plt.title("Tip vs Total Bill, with Regression Line")
plt.show()
plt.close()

g = sns.lmplot(data=df, x="total_bill", y="tip", hue="smoker", height=5)
g.figure.suptitle("Separate Regression Line per Smoker Group", y=1.02)
plt.show()
plt.close("all")

## 14. Seaborn Categorical Plots

`sns.catplot()` is the figure-level umbrella over every categorical plot type — pass `kind="bar"`, `"box"`, `"violin"`, `"strip"`, `"swarm"`, `"count"`, or `"point"` to switch chart type while keeping the same faceting (`col=`/`row=`) interface.

In [ ]:
g = sns.catplot(data=df, x="day", y="total_bill", kind="box", col="time", height=4)
g.figure.suptitle("Total Bill by Day, Faceted by Time", y=1.05)
plt.show()
plt.close("all")

g2 = sns.catplot(data=df, x="day", y="tip", kind="point", hue="smoker", height=4)
plt.show()
plt.close("all")

## 15. Seaborn Facet Grids

`sns.FacetGrid` is the lower-level machinery behind every figure-level function (`displot`, `relplot`, `catplot`, `lmplot`) — use it directly when you want to facet a plot type that doesn't have its own figure-level wrapper.

In [ ]:
g = sns.FacetGrid(df, col="time", row="smoker", height=3.5)
g.map_dataframe(sns.scatterplot, x="total_bill", y="tip")
g.add_legend()
g.figure.suptitle("Custom FacetGrid: Total Bill vs Tip", y=1.03)
plt.show()
plt.close("all")

# relplot is the figure-level, faceted version of scatterplot/lineplot
g2 = sns.relplot(data=df, x="total_bill", y="tip", col="day", hue="time", height=3, col_wrap=2)
plt.show()
plt.close("all")

## 16. Seaborn Styling and Themes

- `sns.set_style()` — the overall look: `"white"`, `"dark"`, `"whitegrid"`, `"darkgrid"`, `"ticks"`.
- `sns.set_palette()` / the `palette=` argument — the color scheme for categorical data.
- `sns.set_context()` — scales font/line sizes for `"paper"`, `"notebook"` (default), `"talk"`, or `"poster"`.
- `sns.despine()` — removes the top/right plot borders for a cleaner look.

In [ ]:
print(sns.axes_style())          # current style parameters
print(sns.color_palette("pastel").as_hex())   # preview a palette as hex codes

with sns.axes_style("darkgrid"):
    sns.scatterplot(data=df, x="total_bill", y="tip", hue="day", palette="pastel")
    plt.title("darkgrid style + pastel palette")
    plt.show()
    plt.close()

sns.set_style("ticks")
ax = sns.scatterplot(data=df, x="total_bill", y="tip")
sns.despine()   # removes the top and right spines
plt.title("ticks style, despined")
plt.show()
plt.close()

sns.set_style("whitegrid")   # reset to the theme used for the rest of the notebook

## 17. Seaborn Built-in Datasets

Seaborn ships a small catalog of example datasets (`"tips"`, `"iris"`, `"titanic"`, `"flights"`, `"penguins"`, ...) via `sns.load_dataset(name)`, fetched from the seaborn-data GitHub repo on first use. This is how almost every seaborn tutorial and the official docs teach the library — but it **requires an internet connection**, so the cell below falls back to a local synthetic dataset if the download fails.

In [ ]:
try:
    iris = sns.load_dataset("iris")
    print("Loaded real 'iris' dataset from seaborn-data")
except Exception as e:
    print(f"No internet access ({type(e).__name__}) - using a synthetic stand-in instead")
    rng = np.random.default_rng(7)
    species = np.repeat(["setosa", "versicolor", "virginica"], 50)
    iris = pd.DataFrame({
        "sepal_length": rng.normal(5.8, 0.8, 150),
        "sepal_width": rng.normal(3.0, 0.4, 150),
        "petal_length": rng.normal(3.7, 1.7, 150),
        "petal_width": rng.normal(1.2, 0.7, 150),
        "species": species,
    })

print(iris.head())

sns.scatterplot(data=iris, x="sepal_length", y="sepal_width", hue="species")
plt.title("Iris Dataset")
plt.show()
plt.close()

## 18. Seaborn Joint Plot

`sns.jointplot()` combines a central bivariate plot (scatter, by default) with the marginal distribution of each variable along the top and right edges — a compact way to see both the relationship and each variable's individual spread at once.

In [ ]:
g = sns.jointplot(data=df, x="total_bill", y="tip")
g.figure.suptitle("Joint Plot: Scatter + Marginal Histograms", y=1.02)
plt.show()
plt.close("all")

g2 = sns.jointplot(data=df, x="total_bill", y="tip", kind="hex")
plt.show()
plt.close("all")

g3 = sns.jointplot(data=df, x="total_bill", y="tip", kind="kde", fill=True)
plt.show()
plt.close("all")